In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.tsa.stattools import adfuller

# 1. Load data and parse dates exactly as structured
df = pd.read_csv('../data/BrentOilPrices.csv')
df['Date'] = pd.to_datetime(df['Date'], format='%d-%b-%y')
df = df.sort_values('Date').reset_index(drop=True)

# 2. Calculate daily log returns to stabilize variance
df['Log_Price'] = np.log(df['Price'])
df['Log_Return'] = df['Log_Price'].diff()

# 3. Print exploratory shapes and null checks
print(f"Dataset covers: {df['Date'].min().strftime('%Y-%m-%d')} to {df['Date'].max().strftime('%Y-%m-%d')}")
print(f"Total trading days: {len(df)}")

# 4. Check for Stationarity using the Augmented Dickey-Fuller (ADF) test
def test_stationarity(series, name):
    result = adfuller(series.dropna())
    print(f"\n--- ADF Test for {name} ---")
    print(f"ADF Statistic: {result[0]:.4f}")
    print(f"p-value: {result[1]:.4e}")
    print(f"Critical Values:")
    for key, value in result[4].items():
        print(f"   {key}: {value:.4f}")
    if result[1] <= 0.05:
        print("Conclusion: Stationary (Reject Null Hypothesis)")
    else:
        print("Conclusion: Non-Stationary (Fail to Reject Null Hypothesis)")

test_stationarity(df['Price'], "Raw Price")
test_stationarity(df['Log_Return'], "Log Returns")

# 5. Plot Price Trends and Volatility Clustering
fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True)

# Top subplot: Raw Pricing Trends
sns.lineplot(data=df, x='Date', y='Price', ax=axes[0], color='navy')
axes[0].set_title('Historical Brent Crude Oil Prices (USD/Barrel)', fontsize=14)
axes[0].set_ylabel('Price (USD)')

# Bottom subplot: Volatility Patterns via Log Returns
sns.lineplot(data=df, x='Date', y='Log_Return', ax=axes[1], color='crimson', alpha=0.6)
axes[1].set_title('Daily Log Returns (Volatility Clustering)', fontsize=14)
axes[1].set_ylabel('Log Return Log(t) - Log(t-1)')

plt.tight_layout()
plt.show()
